<center>
    <img src="https://rockborne.com/wp-content/uploads/2021/07/LandingPage-Header-RED-CENTRE.jpg" width="900" alt="logo"  />
</center>

# Practical Data Quality: San Francisco Salaries (Student Version)

## About this notebook
This is the **data-quality companion** to the SF Salaries pandas practical. There you manipulated the data and spotted a few odd things (missing values, zero or negative pay, columns that looked empty). Here you'll investigate those issues properly using the techniques from the **Data Cleaning** walkthrough: inspecting data, finding and quantifying missing values, spotting redundant and inconsistent columns, validating the numbers, and finally producing a cleaned dataset.

## About the exercises
Each exercise is made of two cells:

1. **The question** (a markdown cell like this one).
2. **`# Your turn`**: an empty code cell for you to write your solution.

Have a go at each one yourself. Later exercises reuse variables created in earlier ones, so work through them in order.

## About the data
The same `Salaries.csv` from the pandas practical, in [`../datasets/`](../datasets/): about 148,654 rows of San Francisco municipal pay records for 2011 to 2014. As you'll discover, the 2011 batch was recorded differently from later years, which is the source of several quality issues.

---
# Part 0: Setup and loading the data

In [ ]:
import pandas as pd
## Loading data from S3
from config import session_datasets

### Exercise 0: Load the data

Read `../datasets/Salaries.csv` into a DataFrame called `sal`.

In [ ]:
# Your turn. Write your solution here:


---
# Part 1: First inspection
A quick health check before trusting any numbers.

### Exercise 1.1: Shape and structure

How many rows and columns are there, and what is the dtype and non-null count of each column?

In [ ]:
# Your turn. Write your solution here:


### Exercise 1.2: Suspicious summary statistics

Run `describe()` on the numeric columns. Look at the **minimum** values: which columns have values that shouldn't be possible for pay?

> *Hint: Negative pay and a minimum of zero are both red flags worth investigating.*

In [ ]:
# Your turn. Write your solution here:


---
# Part 2: Identifying missing data

### Exercise 2.1: Missing values per column

Count how many missing (null) values each column has.

> *Hint: `.isnull()` gives a boolean DataFrame; `.sum()` counts the `True`s per column.*

In [ ]:
# Your turn. Write your solution here:


### Exercise 2.2: As a percentage, sorted

Express the missing counts as a percentage of all rows, sorted from worst to best.

> *Hint: `.isnull().mean()` gives the fraction missing per column.*

In [ ]:
# Your turn. Write your solution here:


### Exercise 2.3: Completely empty columns

Which columns are **100% empty** (every value missing)?

> *Hint: `sal[c].isnull().all()` is True when the whole column is missing.*

In [ ]:
# Your turn. Write your solution here:


### Exercise 2.4: Where do the missing Benefits come from?

`Benefits` has tens of thousands of missing values. Break the missing count down **by `Year`**. What do you notice?

> *Hint: Group by Year, then count nulls within each group. Almost all of them are in one year.*

In [ ]:
# Your turn. Write your solution here:


---
# Part 3: Redundant and constant columns
A column that never changes (or is always empty) carries no information.

### Exercise 3.1: Find the constant columns

Which columns contain only a single distinct value (ignoring missing values)?

> *Hint: `nunique(dropna=True)` counts distinct non-null values; 1 or 0 means constant.*

In [ ]:
# Your turn. Write your solution here:


### Exercise 3.2: Drop the dead weight

`Notes` and `Status` are empty and `Agency` is always the same. Create `sal2` without those three columns (keep `sal` intact).

In [ ]:
# Your turn. Write your solution here:


---
# Part 4: Inconsistent text
Look closely at `EmployeeName` and `JobTitle`: the 2011 records use UPPERCASE while later years use Mixed Case. That inconsistency makes the same job look like two different jobs.

### Exercise 4.1: Confirm the case problem by year

For each `Year`, how many `EmployeeName` values are entirely uppercase? This should pinpoint the odd year.

> *Hint: `.str.isupper()` returns True for fully uppercase strings; group that by Year.*

In [ ]:
# Your turn. Write your solution here:


### Exercise 4.2: Standardise the case

Standardise `JobTitle` to a single case (uppercase) in `sal2`. How many distinct job titles were there before, and how many after? Why does the number drop?

> *Hint: `.str.upper()` (with a `.str.strip()` to remove stray spaces) merges titles that differed only by capitalisation.*

In [ ]:
# Your turn. Write your solution here:


---
# Part 5: Validity and consistency checks
Are the numbers internally consistent, and are they plausible?

### Exercise 5.1: Negative pay

Show the rows where `TotalPayBenefits` is negative.

In [ ]:
# Your turn. Write your solution here:


### Exercise 5.2: Zero or negative total pay

How many rows have `TotalPay` of zero or less?

> *Hint: These are likely people who left, were not actually paid, or data-entry errors.*

In [ ]:
# Your turn. Write your solution here:


### Exercise 5.3: Does the pay add up?

Check whether `BasePay + OvertimePay + OtherPay` equals `TotalPay`. Count the rows that differ by more than a cent.

> *Hint: If the count is 0, the components are internally consistent (a good sign).*

In [ ]:
# Your turn. Write your solution here:


### Exercise 5.4: Does TotalPayBenefits add up?

Check whether `TotalPay + Benefits` equals `TotalPayBenefits`. Treat missing `Benefits` as 0. Count the mismatches.

> *Hint: `.fillna(0)` lets the 2011 rows (no benefits recorded) pass the check.*

In [ ]:
# Your turn. Write your solution here:


---
# Part 6: Handling the missing values
Now decide what to *do* about the gaps. The right choice depends on what the gap means.

### Exercise 6.1: Inspect the missing BasePay

How many rows have a missing `BasePay`, and what do a few of them look like?

In [ ]:
# Your turn. Write your solution here:


### Exercise 6.2: Fill the missing Benefits

Missing `Benefits` mostly means benefits weren't recorded (2011), so 0 is a reasonable fill. Fill the missing `Benefits` with 0 in `sal2` and confirm none remain.

> *Hint: `.fillna(0)` replaces missing values with zero.*

In [ ]:
# Your turn. Write your solution here:


### Exercise 6.3: Fill the missing BasePay

For `BasePay`, fill the gaps with the column **median** (more robust than the mean for skewed pay). Confirm none remain.

> *Hint: `.fillna(sal2['BasePay'].median())`.*

In [ ]:
# Your turn. Write your solution here:


---
# Part 7: Duplicates

### Exercise 7.1: Exact duplicate rows

Are there any fully duplicated rows in `sal2`?

In [ ]:
# Your turn. Write your solution here:


### Exercise 7.2: Same name, same year

How many rows share the same `EmployeeName` and `Year`? Is that necessarily a data error?

> *Hint: Common names exist, so a repeated (name, year) is not automatically a duplicate person. There's no unique person id here to be sure.*

In [ ]:
# Your turn. Write your solution here:


---
# Part 8: Produce a cleaned dataset
Bring your decisions together into one clean table and save it.

### Exercise 8: Assemble and export the clean data

Starting from `sal2` (constant columns already dropped, text standardised, `Benefits`/`BasePay` filled), remove the invalid rows where `TotalPay <= 0`, then save the result to `../datasets/Salaries_clean.csv`. Report the final shape.

In [ ]:
# Your turn. Write your solution here:


## Summary of cleaning decisions
- **Dropped** `Notes` and `Status` (100% empty) and `Agency` (constant).
- **Standardised** `JobTitle` case, collapsing titles that differed only by capitalisation.
- **Filled** missing `Benefits` with 0 (not recorded, mostly 2011) and missing `BasePay` with the median.
- **Removed** rows with `TotalPay <= 0` as implausible.
- **Kept** same-name/same-year rows: without a unique person id we can't treat them as duplicates.

Every choice above is a judgement call. In a real project you would document each one, exactly as here.